<h3 style="color:#6FA8DC; font-weight:bold">02 — Feature Scaling</h3>

<h5 style="color:#78B89A; font-weight:bold;">Why do we need Feature Scaling and how does it work?</h5>

Feature Scaling is an important part of **Feature Transformation** in Feature Engineering.

It changes the numerical range of features so that different columns become comparable.

In this notebook, we will understand:

- What Feature Scaling is
- Why Feature Scaling is needed
- When scaling is required
- Types of Feature Scaling
- Standardisation in detail
- StandardScaler practically
- Graphical comparison before and after scaling
- Min-Max Scaling
- Robust Scaling
- MaxAbs Scaling
- Normalisation vs Standardisation
- Important practical rules

We will use the **Social Network Ads** dataset for practical examples.

<div style="border-top:black 2px solid"></div>

# 1. What is Feature Scaling?

Feature Scaling is the process of converting numerical features into a comparable scale.

Suppose our dataset contains:

| Feature | Typical Values |
|---|---:|
| Age | 18 to 60 |
| Estimated Salary | 15,000 to 150,000 |

Both columns contain useful information, but their numerical ranges are very different.

If an algorithm uses distance or mathematical calculations, the salary column may dominate simply because its values are numerically larger.

### Simple definition

> **Feature Scaling means transforming numerical features so that their values lie within a comparable range or follow a comparable distribution.**

# 2. Why do we need Feature Scaling?

## Problem without scaling

Imagine two customers:

- Customer A: Age = 25, Salary = 40,000
- Customer B: Age = 35, Salary = 60,000

The difference is:

```text
Age difference = 10
Salary difference = 20,000
```

For a distance-based algorithm, salary may contribute much more to the distance because its numerical values are much larger.

The algorithm may unintentionally give more importance to salary than age.

### Real-life ML example

In the Social Network Ads dataset, we want to predict whether a person will purchase a product using:

- Age
- Estimated Salary

Algorithms such as KNN calculate distances between customers.

If salary is not scaled, the distance calculation may be dominated by salary.

Scaling helps both features contribute more fairly.

## 2.1 Algorithms that commonly require or benefit from scaling

### Distance-based algorithms

- K-Nearest Neighbours (KNN)
- K-Means Clustering
- DBSCAN

### Gradient-based algorithms

- Logistic Regression
- Linear Regression in many practical situations
- Support Vector Machines
- Neural Networks

### Algorithms based on variance or projections

- Principal Component Analysis (PCA)

### Algorithms that generally do not require scaling

Tree-based algorithms usually do not depend on distance in the same way:

- Decision Tree
- Random Forest
- Gradient Boosting Trees
- XGBoost and similar tree-based models

### Important

Scaling is not automatically compulsory for every algorithm.

It depends on how the algorithm learns patterns.

# 3. Types of Feature Scaling

The main types are:

1. Standardisation / Z-score Scaling
2. Min-Max Scaling / Normalisation
3. Robust Scaling
4. MaxAbs Scaling
5. Vector Normalisation

We will study each one, but Standardisation will be explained in the greatest detail.

<div style="border-top:black 2px solid"></div>

# 4. Standardisation

## What is Standardisation?

Standardisation transforms a numerical feature so that:

- Mean becomes approximately 0
- Standard deviation becomes approximately 1

It is also called:

- Z-score scaling
- Standard scaling

### Formula

\[
z = \frac{x - \mu}{\sigma}
\]

Where:

- `x` = original value
- `μ` = mean of the feature
- `σ` = standard deviation of the feature
- `z` = standardised value

### Interpretation

After standardisation:

- `z = 0` → value is equal to the mean
- `z = 1` → value is one standard deviation above the mean
- `z = -1` → value is one standard deviation below the mean

Standardisation does **not** force every value between 0 and 1.

## 4.1 Simple numerical example

Suppose the ages are:

```text
20, 25, 30, 35, 40
```

Mean:

```text
30
```

Assume standard deviation:

```text
7.07
```

For age = 40:

\[
z = \frac{40 - 30}{7.07}
\]

The result is approximately:

```text
+1.41
```

This means age 40 is approximately 1.41 standard deviations above the mean.

## 4.2 Standardisation visual diagram

The idea is:

```text
Original Feature
       |
       | subtract mean
       v
Centered around 0
       |
       | divide by standard deviation
       v
Standardised Feature
(mean ≈ 0, std ≈ 1)
```

The shape of the distribution is generally preserved; the location and scale are changed.

In [ ]:
# Visual explanation of standardisation

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(10, 90, 200)
mean = 50
std = 10

z = (x - mean) / std

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 4))

ax1.plot(x, np.exp(-0.5 * ((x - mean) / std) ** 2))
ax1.axvline(mean, linestyle='--')
ax1.set_title('Original Scale')
ax1.set_xlabel('Original Values')
ax1.set_ylabel('Relative Density')

ax2.plot(z, np.exp(-0.5 * z ** 2))
ax2.axvline(0, linestyle='--')
ax2.set_title('After Standardisation')
ax2.set_xlabel('Z-scores')
ax2.set_ylabel('Relative Density')

plt.tight_layout()
plt.show()

### What does the graph show?

Before standardisation:

- Values are centered around the original mean.
- The x-axis uses the original units.

After standardisation:

- The center moves to approximately 0.
- The spread is expressed in standard deviation units.
- The overall distribution shape remains similar.

This is why standardisation is often described as changing the **location and scale**, not fundamentally changing the distribution.

<div style="border-top:black 2px solid"></div>

# 5. Loading the Social Network Ads Dataset

The dataset contains information about users and whether they purchased a product after seeing an advertisement.

Columns:

- `User ID`
- `Gender`
- `Age`
- `EstimatedSalary`
- `Purchased`

For scaling, we will use:

- `Age`
- `EstimatedSalary`

We will not use:

- `User ID` because it is an identifier, not a meaningful predictive feature.
- `Gender` directly because it is categorical.
- `Purchased` because it is the target variable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Social_Network_Ads.csv')

df.head()

In [ ]:
df = df.iloc[:, 2:]

df.sample(5)

### Separating input features and target

In [ ]:
X = df.drop('Purchased', axis=1)
y = df['Purchased']

X.head(), y.head()

<div style="border-top:black 2px solid"></div>

# 6. Train-Test Split

Before scaling, we should split the dataset into training and testing sets.

### Why?

The scaler learns values such as:

- Mean
- Standard deviation
- Minimum
- Maximum
- Median

These values must be learned only from the training data.

If we calculate them using the complete dataset, information from the test set may leak into the training process.

This is called **data leakage**.

### Correct order

```text
Dataset
   ↓
Train-Test Split
   ↓
Fit scaler only on training data
   ↓
Transform training and testing data
```

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0
)

X_train.shape, X_test.shape

<div style="border-top:black 2px solid"></div>

# 7. Applying StandardScaler

`StandardScaler` is available inside:

```python
sklearn.preprocessing
```

### Workflow

1. Create the scaler object.
2. Fit it on training data.
3. Transform training data.
4. Transform testing data.

### Important difference

- `fit()` → learns parameters
- `transform()` → applies the learned transformation
- `fit_transform()` → performs both operations

For the test set, we should use only `transform()`.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn mean and standard deviation from training data
scaler.fit(X_train)

# Transform both datasets using training parameters
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Mean learned by the scaler
scaler.mean_

In [ ]:
# Original training data
X_train.head()

In [ ]:
# Scaled training data
X_train_scaled[:5]

## 7.1 Converting scaled arrays back into DataFrames

`StandardScaler` returns a NumPy array.

For easier reading, we convert it back into a DataFrame and restore the column names.

In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.head()

<div style="border-top:black 2px solid"></div>

# 8. Checking the effect of Standardisation

## Before scaling vs After scaling

Before scaling:

- Age has values approximately between 18 and 60.
- EstimatedSalary has values in thousands.

After scaling:

- Both columns are expressed in standard deviation units.
- Both are centered around 0.
- Both have a comparable spread.

In [ ]:
np.round(X_train.describe(), 2)

In [ ]:
np.round(X_train_scaled.describe(), 2)

### Important observation

The actual units change.

For example:

- Age is no longer measured in years.
- Salary is no longer measured in currency.

Instead, values represent how far an observation is from the feature mean in standard deviation units.

<div style="border-top:black 2px solid"></div>

# 9. Graphical Comparison

## 9.1 Scatterplot before and after scaling

Scaling does not change the relative ordering of observations. It changes the numerical axes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(
    X_train['Age'],
    X_train['EstimatedSalary']
)
ax1.set_title('Before Scaling')
ax1.set_xlabel('Age')
ax1.set_ylabel('Estimated Salary')

ax2.scatter(
    X_train_scaled['Age'],
    X_train_scaled['EstimatedSalary']
)
ax2.set_title('After Standard Scaling')
ax2.set_xlabel('Standardised Age')
ax2.set_ylabel('Standardised Salary')

plt.tight_layout()
plt.show()

## 9.2 Distribution comparison

Standardisation generally preserves the shape of a feature's distribution while changing its center and spread.

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.set_title('Before Scaling')
sns.kdeplot(X_train['Age'], ax=ax1, label='Age')
sns.kdeplot(X_train['EstimatedSalary'], ax=ax1, label='Estimated Salary')
ax1.legend()

ax2.set_title('After Standard Scaling')
sns.kdeplot(X_train_scaled['Age'], ax=ax2, label='Age')
sns.kdeplot(X_train_scaled['EstimatedSalary'], ax=ax2, label='Estimated Salary')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.set_title('Age Distribution Before Scaling')
sns.kdeplot(X_train['Age'], ax=ax1, fill=True)

ax2.set_title('Age Distribution After Scaling')
sns.kdeplot(X_train_scaled['Age'], ax=ax2, fill=True)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.set_title('Salary Distribution Before Scaling')
sns.kdeplot(X_train['EstimatedSalary'], ax=ax1, fill=True)

ax2.set_title('Salary Distribution After Scaling')
sns.kdeplot(X_train_scaled['EstimatedSalary'], ax=ax2, fill=True)

plt.tight_layout()
plt.show()

<div style="border-top:black 2px solid"></div>

# 10. Min-Max Scaling

## What is Min-Max Scaling?

Min-Max Scaling transforms values into a fixed range, usually **0 to 1**.

### Formula

\[
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

### Properties

- Minimum value becomes 0.
- Maximum value becomes 1.
- Other values lie between 0 and 1, provided they are within the learned range.

### When is it useful?

- When a fixed range is desirable.
- In some neural network applications.
- When features have different units.
- When the algorithm benefits from bounded values.

### Limitation

Min-Max scaling is sensitive to outliers because minimum and maximum values determine the transformation.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

X_train_minmax = pd.DataFrame(
    X_train_minmax,
    columns=X_train.columns,
    index=X_train.index
)

X_train_minmax.head()

<div style="border-top:black 2px solid"></div>

# 11. Robust Scaling

## What is Robust Scaling?

Robust Scaling uses statistics that are less affected by outliers:

- Median
- Interquartile Range (IQR)

### Formula

\[
x' = \frac{x - Median}{IQR}
\]

where:

\[
IQR = Q3 - Q1
\]

### Why use it?

If a feature contains extreme values, mean and standard deviation may be strongly affected.

Median and IQR are more resistant to extreme observations.

### Real-life example

Transaction amounts may contain a few extremely large purchases.

Robust Scaling can be useful when those extreme values should remain in the dataset but should not dominate the scaling parameters.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()

X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

X_train_robust = pd.DataFrame(
    X_train_robust,
    columns=X_train.columns,
    index=X_train.index
)

X_train_robust.head()

<div style="border-top:black 2px solid"></div>

# 12. MaxAbs Scaling

## What is MaxAbs Scaling?

MaxAbs Scaling divides each feature by its maximum absolute value.

### Formula

\[
x' = \frac{x}{\max(|x|)}
\]

### Properties

- Values are generally scaled between -1 and 1.
- It preserves zero values.
- It is useful for sparse data because it does not center the data.

### Common use

It may be useful when working with sparse matrices, such as some text-based datasets.

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

maxabs_scaler = MaxAbsScaler()

X_train_maxabs = maxabs_scaler.fit_transform(X_train)
X_test_maxabs = maxabs_scaler.transform(X_test)

X_train_maxabs = pd.DataFrame(
    X_train_maxabs,
    columns=X_train.columns,
    index=X_train.index
)

X_train_maxabs.head()

<div style="border-top:black 2px solid"></div>

# 13. Normalisation vs Standardisation

These two terms are often confused.

## Standardisation

- Usually applied column-wise.
- Centers data around mean 0.
- Divides by standard deviation.
- Produces z-scores.

## Normalisation

In Machine Learning, normalisation often means scaling each individual observation/vector to have a unit norm.

For example, a row containing:

```text
[3, 4]
```

has Euclidean norm:

\[
\sqrt{3^2 + 4^2} = 5
\]

After L2 normalisation:

```text
[3/5, 4/5] = [0.6, 0.8]
```

### Important

- StandardScaler works feature-wise/column-wise.
- Normalizer works row-wise/sample-wise.

In [ ]:
from sklearn.preprocessing import Normalizer

sample = np.array([
    [3, 4],
    [5, 12]
])

normalizer = Normalizer()

normalizer.transform(sample)

<div style="border-top:black 2px solid"></div>

# 14. Comparison of Scaling Techniques

| Technique | Main Idea | Range / Result | Sensitive to Outliers? |
|---|---|---|---|
| Standardisation | Subtract mean and divide by standard deviation | Mean ≈ 0, std ≈ 1 | Yes, to some extent |
| Min-Max Scaling | Use minimum and maximum | Usually 0 to 1 | Yes |
| Robust Scaling | Use median and IQR | No fixed range | Less sensitive |
| MaxAbs Scaling | Divide by maximum absolute value | Usually -1 to 1 | Yes |
| Normalisation | Scale each row/vector by its norm | Unit norm | Depends on data |

### Quick memory trick

- **StandardScaler** → Mean 0, Standard Deviation 1
- **MinMaxScaler** → 0 to 1
- **RobustScaler** → Median and IQR
- **MaxAbsScaler** → Maximum absolute value
- **Normalizer** → Each row gets unit length

# 15. Which scaler should we choose?

### Use StandardScaler when:

- Data is reasonably well-behaved.
- You want mean 0 and standard deviation 1.
- You are using Logistic Regression, SVM, KNN, or PCA.

### Use MinMaxScaler when:

- You need a fixed range.
- You want values between 0 and 1.
- Extreme values are not a major issue.

### Use RobustScaler when:

- The dataset contains significant outliers.
- You want scaling based on median and IQR.

### Use MaxAbsScaler when:

- Data is sparse.
- Preserving zero values is important.

### Use Normalizer when:

- The magnitude of each row/vector matters less than its direction.
- You are working with text vectors or similar high-dimensional representations.

<div style="border-top:black 2px solid"></div>

# 16. Important Practical Rules

1. Split the data before fitting the scaler.
2. Fit the scaler only on training data.
3. Use the same fitted scaler to transform the test data.
4. Never fit a separate scaler on the test set.
5. Do not scale the target variable casually in classification problems.
6. Do not scale ID columns just because they are numerical.
7. Scaling does not remove outliers.
8. Scaling does not convert categorical data into numerical data.
9. Scaling changes units but generally preserves the order of values.
10. Use a Pipeline in real Machine Learning projects to avoid leakage and keep preprocessing organised.

### Example of data leakage

Incorrect:

```python
scaler.fit_transform(X)
train_test_split(X_scaled, y)
```

Correct:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y)

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

# Final Revision

## Feature Scaling

Feature Scaling makes numerical features comparable.

### Why?

Because some algorithms are affected by feature magnitude.

### Main types

1. Standardisation
2. Min-Max Scaling
3. Robust Scaling
4. MaxAbs Scaling
5. Normalisation

### Standardisation

\[
z = \frac{x - \mu}{\sigma}
\]

Result:

- Mean approximately 0
- Standard deviation approximately 1

### Complete workflow

```text
Load Data
   ↓
Separate X and y
   ↓
Train-Test Split
   ↓
Fit scaler on X_train
   ↓
Transform X_train
   ↓
Transform X_test
   ↓
Train Machine Learning Model
```

### Final memory trick

**Split → Fit on Train → Transform Train → Transform Test**

<div style="border-top:black 2px solid"></div>

## One-line takeaway

> **Feature Scaling changes the numerical scale of features so that Machine Learning algorithms can learn without being unfairly influenced by large numerical ranges.**